# 214 — Ingest Layer 2 into Neo4j

Loads the MAS Notice 626 parsed CSVs (`data/layer_2/extracted/`) into Neo4j as:

```
(Regulation)-[:HAS_SECTION]->(Section)-[:NEXT_SECTION]->(Section)
                                  |
                                  +-[:HAS_REQUIREMENT]->(Requirement)
                                                          |
                                                          +-[:DEFINES_THRESHOLD]->(Threshold)
                                                          +-[:HAS_CHUNK]->(Chunk)-[:NEXT_CHUNK]->(Chunk)
                                                          +-[:CROSS_REFERENCES]->(Requirement)
```

No embeddings here — notebook 215 does that. This notebook is idempotent: re-running it will not duplicate nodes or edges (all MERGE).

**Prerequisites**
- `.env` with `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`
- Extracted CSVs present in `data/layer_2/extracted/`

## Imports and connection

In [1]:
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

REPO = Path.cwd().parent
sys.path.insert(0, str(REPO))
load_dotenv(REPO / '.env')

from src.graph.connection import Neo4jConnection

EX = REPO / 'data' / 'layer_2' / 'extracted'

conn = Neo4jConnection().connect()
print('Connected to', conn._uri)

Connected to neo4j+s://692d7258.databases.neo4j.io


## Helpers

In [2]:
BATCH_SIZE = 500

def df_records(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False).to_dict(orient='records')

def batch_run(cypher, records, batch_size=BATCH_SIZE):
    total = 0
    for i in range(0, len(records), batch_size):
        conn.run_query(cypher, {'records': records[i:i + batch_size]})
        total += len(records[i:i + batch_size])
    return total

## Optional: reset Layer 2

Flip `RESET = True` to wipe Layer 2 before reloading. Layer 1 and Layer 3 are untouched.

In [3]:
RESET = False
if RESET:
    for label in ('Chunk', 'Threshold', 'Requirement', 'Section', 'Regulation'):
        conn.run_query(f'MATCH (n:{label}) DETACH DELETE n')
    print('Layer 2 wiped.')
else:
    print('Skipped reset (RESET=False).')

Skipped reset (RESET=False).


## Uniqueness constraints

In [4]:
CONSTRAINTS = [
    'CREATE CONSTRAINT regulation_id   IF NOT EXISTS FOR (n:Regulation)  REQUIRE n.regulation_id  IS UNIQUE',
    'CREATE CONSTRAINT section_id      IF NOT EXISTS FOR (n:Section)     REQUIRE n.section_id     IS UNIQUE',
    'CREATE CONSTRAINT requirement_id  IF NOT EXISTS FOR (n:Requirement) REQUIRE n.requirement_id IS UNIQUE',
    'CREATE CONSTRAINT threshold_id    IF NOT EXISTS FOR (n:Threshold)   REQUIRE n.threshold_id   IS UNIQUE',
    'CREATE CONSTRAINT chunk_id        IF NOT EXISTS FOR (n:Chunk)       REQUIRE n.chunk_id       IS UNIQUE',
]
for c in CONSTRAINTS:
    conn.run_query(c)
print(f'Applied {len(CONSTRAINTS)} constraints.')

Applied 5 constraints.


## Load nodes

In [5]:
# Regulation
records = df_records(EX / 'regulations.csv')
n = batch_run(
    'UNWIND $records AS row MERGE (n:Regulation {regulation_id: row.regulation_id}) SET n += row',
    records,
)
print(f'Regulation:  {n:>4} nodes')

# Section
records = df_records(EX / 'sections.csv')
n = batch_run(
    'UNWIND $records AS row MERGE (n:Section {section_id: row.section_id}) SET n += row',
    records,
)
print(f'Section:     {n:>4} nodes')

# Requirement
records = df_records(EX / 'requirements.csv')
n = batch_run(
    'UNWIND $records AS row MERGE (n:Requirement {requirement_id: row.requirement_id}) SET n += row',
    records,
)
print(f'Requirement: {n:>4} nodes')

# Threshold
records = df_records(EX / 'thresholds.csv')
n = batch_run(
    'UNWIND $records AS row MERGE (n:Threshold {threshold_id: row.threshold_id}) SET n += row',
    records,
)
print(f'Threshold:   {n:>4} nodes')

# Chunk
records = df_records(EX / 'chunks.csv')
n = batch_run(
    'UNWIND $records AS row MERGE (n:Chunk {chunk_id: row.chunk_id}) SET n += row',
    records,
)
print(f'Chunk:       {n:>4} nodes')

Regulation:     1 nodes
Section:       15 nodes
Requirement:  147 nodes
Threshold:     23 nodes
Chunk:        268 nodes


## Build edges

All edges can be derived from the node properties — no separate link CSVs needed for Layer 2.

In [6]:
# HAS_SECTION: Regulation -> Section
r = conn.run_query('''
    MATCH (r:Regulation), (s:Section)
    WHERE s.regulation_id = r.regulation_id
    MERGE (r)-[:HAS_SECTION]->(s)
    RETURN count(*) AS n
''')
print(f'HAS_SECTION:       {r[0]["n"]:>4} edges')

# NEXT_SECTION: Section -> Section (same regulation, numeric section ordering)
r = conn.run_query('''
    MATCH (s:Section)
    WITH s.regulation_id AS reg,
         s,
         toInteger(s.section_number) AS sn
    ORDER BY reg, sn
    WITH reg, collect(s) AS ordered
    UNWIND range(0, size(ordered) - 2) AS i
    WITH ordered[i] AS a, ordered[i + 1] AS b
    MERGE (a)-[:NEXT_SECTION]->(b)
    RETURN count(*) AS n
''')
print(f'NEXT_SECTION:      {r[0]["n"]:>4} edges')

# HAS_REQUIREMENT: Section -> Requirement
r = conn.run_query('''
    MATCH (s:Section), (req:Requirement)
    WHERE req.section_id = s.section_id
    MERGE (s)-[:HAS_REQUIREMENT]->(req)
    RETURN count(*) AS n
''')
print(f'HAS_REQUIREMENT:   {r[0]["n"]:>4} edges')

# DEFINES_THRESHOLD: Requirement -> Threshold (match on paragraph + regulation_id)
r = conn.run_query('''
    MATCH (req:Requirement), (t:Threshold)
    WHERE req.regulation_id = t.regulation_id
      AND req.paragraph = t.paragraph
    MERGE (req)-[:DEFINES_THRESHOLD]->(t)
    RETURN count(*) AS n
''')
print(f'DEFINES_THRESHOLD: {r[0]["n"]:>4} edges')

# HAS_CHUNK: Requirement -> Chunk
r = conn.run_query('''
    MATCH (req:Requirement), (c:Chunk)
    WHERE req.regulation_id = c.regulation_id
      AND req.paragraph = c.paragraph
    MERGE (req)-[:HAS_CHUNK]->(c)
    RETURN count(*) AS n
''')
print(f'HAS_CHUNK:         {r[0]["n"]:>4} edges')

# NEXT_CHUNK: Chunk -> Chunk (within the same paragraph, ordered by chunk_index)
r = conn.run_query('''
    MATCH (c:Chunk)
    WITH c.regulation_id AS reg,
         c.paragraph AS para,
         c,
         toInteger(c.chunk_index) AS idx
    ORDER BY reg, para, idx
    WITH reg, para, collect(c) AS ordered
    UNWIND range(0, size(ordered) - 2) AS i
    WITH ordered[i] AS a, ordered[i + 1] AS b
    MERGE (a)-[:NEXT_CHUNK]->(b)
    RETURN count(*) AS n
''')
print(f'NEXT_CHUNK:        {r[0]["n"]:>4} edges')

# CROSS_REFERENCES: Requirement -> Requirement (both paragraphs in same regulation)
records = df_records(EX / 'cross_references.csv')
n = batch_run(
    '''
    UNWIND $records AS row
    MATCH (src:Requirement {regulation_id: row.regulation_id, paragraph: row.source_paragraph})
    MATCH (tgt:Requirement {regulation_id: row.regulation_id, paragraph: row.target_paragraph})
    MERGE (src)-[r:CROSS_REFERENCES]->(tgt)
    SET r.context = row.context
    ''',
    records,
)
print(f'CROSS_REFERENCES:  {n:>4} edges attempted')


HAS_SECTION:         15 edges
NEXT_SECTION:        14 edges
HAS_REQUIREMENT:    146 edges
DEFINES_THRESHOLD:   23 edges
HAS_CHUNK:          267 edges
NEXT_CHUNK:         121 edges
CROSS_REFERENCES:    71 edges attempted


## Verification

In [7]:
print('Layer 2 nodes:')
for row in conn.run_query('''
    MATCH (n)
    WHERE any(lbl IN labels(n) WHERE lbl IN ['Regulation','Section','Requirement','Threshold','Chunk'])
    RETURN labels(n)[0] AS label, count(*) AS n
    ORDER BY n DESC
'''):
    print(f'  {row["label"]:<14} {row["n"]:>4}')

print('\nLayer 2 edges:')
for row in conn.run_query('''
    MATCH ()-[r]->()
    WHERE type(r) IN ['HAS_SECTION','NEXT_SECTION','HAS_REQUIREMENT','DEFINES_THRESHOLD','HAS_CHUNK','NEXT_CHUNK','CROSS_REFERENCES']
    RETURN type(r) AS rel, count(*) AS n
    ORDER BY n DESC
'''):
    print(f'  {row["rel"]:<20} {row["n"]:>4}')

print('\nSpot check — SGD thresholds attached to requirements:')
for row in conn.run_query('''
    MATCH (req:Requirement)-[:DEFINES_THRESHOLD]->(t:Threshold)
    WHERE t.unit = 'SGD'
    RETURN req.paragraph AS para, t.operator + ' ' + t.value + ' ' + t.unit AS threshold
    ORDER BY para LIMIT 10
'''):
    print(f'  {row["para"]:<8} {row["threshold"]}')

Layer 2 nodes:
  Chunk           267
  Requirement     146
  Threshold        23
  Section          15
  Regulation        1

Layer 2 edges:
  HAS_CHUNK             267
  HAS_REQUIREMENT       146
  NEXT_CHUNK            121
  CROSS_REFERENCES       60
  DEFINES_THRESHOLD      23
  HAS_SECTION            15
  NEXT_SECTION           14

Spot check — SGD thresholds attached to requirements:
  11.16    >= 1500 SGD
  11.16    >= 1500 SGD
  11.16    >= 1500 SGD
  11.16    >= 1500 SGD
  11.16    >= 1500 SGD
  11.16    >= 1500 SGD
  11.3     >= 1500 SGD
  11.4     >= 1500 SGD
  11.4     >= 1500 SGD
  11.5     >= 1500 SGD


In [8]:
conn.close()
print('Done.')

Done.
